# Asset Class Trend Following 策略回測 (equityV1-1)

本報告針對 **2019-01-01 - 2026-03-31** 期間進行回測。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backtest_v2 import clean_data, BacktesterV2, calculate_metrics

# 1. 資料讀取與清洗
prices, volumes, code_to_name = clean_data('資料-1.xlsx')

# 2. 設定參數與執行回測
sma, roc, sl, reb = 303, 14, 0.0999, 9
bt = BacktesterV2(prices, volumes, code_to_name)
eq, trades, hold, trades2, daily = bt.run(sma, roc, sl, reb, 'peak', 10)

# 3. 篩選指定期間績效
mask = (eq['日期'] >= '2019-01-01') & (eq['日期'] <= '2026-03-31')
res_p = eq[mask]

# 4. 繪製權益曲線
plt.figure(figsize=(12, 6))
plt.plot(res_p['日期'], res_p['權益'])
plt.title(f'Equity Curve (equityV1-1)')
plt.grid(True)
plt.show()

# 5. 計算績效指標
cagr, mdd, calmar, total_ret = calculate_metrics(res_p)
print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD): {mdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")

# Walk-Forward Analysis 樣板程式 (含自動繪製權益曲線)
periods = [
    ('2019-01-02', '2021-12-31'), 
    ('2019-06-01', '2022-05-31'),
    ('2020-01-02', '2022-12-31'),
    ('2020-06-01', '2023-05-31'),
    ('2021-01-02', '2023-12-31'),
    ('2021-06-01', '2024-05-31'),
    ('2022-01-02', '2024-12-31'),
    ('2022-06-01', '2025-05-31'),
    ('2023-01-02', '2025-12-31'),
]

wfa_results = []
plt.figure(figsize=(12, 6))
for start, end in periods:
    mask_wfa = (eq['日期'] >= start) & (eq['日期'] <= end)
    res_wfa = eq[mask_wfa]
    cagr_wfa, mdd_wfa, calmar_wfa, total_ret_wfa = calculate_metrics(res_wfa)
    wfa_results.append({
        '區間': f"{start} ~ {end}",
        'CAGR': f"{cagr_wfa:.2%}",
        'MaxDD': f"{mdd_wfa:.2%}",
        'Calmar': f"{calmar_wfa:.2f}",
        '總報酬率': f"{total_ret_wfa:.2%}"
    })
    plt.plot(res_wfa['日期'], res_wfa['權益'], label=f"{start} ~ {end}")

plt.title("Walk-Forward Analysis - Equity Curves")
plt.xlabel("日期")
plt.ylabel("權益")
plt.legend()
plt.grid(True)
plt.show()
wfa_df = pd.DataFrame(wfa_results)
print(wfa_df)

# 6. 2026 績效
mask_2026 = (eq['日期'] >= '2026-01-01') & (eq['日期'] <= '2026-03-31')
res_2026 = eq[mask_2026]
cagr_2026, mdd_2026, calmar_2026, total_ret_2026 = calculate_metrics(res_2026)
print("\n2026 績效 (2026.01.01 - 2026.03.31):")
print(f"CAGR: {cagr_2026:.2%}")
print(f"MaxDD: {mdd_2026:.2%}")
print(f"Calmar Ratio: {calmar_2026:.2f}")
